# Baseline results: Hallucination, Overall Metrics, Numeric Tolerance

Compares **Gemini 2.0 Flash**, **Gemini 2.5 Flash**, and **LandingAI** across the 6 PDFs.
Three tables are produced:
- Hallucination rate (empty GT but non-empty prediction)
- Overall correctness/completeness/overall score
- Numeric tolerance score


In [1]:
import json
from pathlib import Path
import pandas as pd

# Resolve experiment-scripts directory (run from repo root or experiment-analysis/)
_candidates = [
    Path.cwd() / 'experiment-scripts',
    (Path.cwd() / '..' / 'experiment-scripts').resolve(),
    Path.cwd(),
]
SCRIPT_BASE = next((p for p in _candidates if p.exists()), None)
if SCRIPT_BASE is None:
    raise FileNotFoundError('Could not find experiment-scripts. Run from repo root or experiment-analysis/.')

BASELINE_RUNS = [
    ('Gemini 2.0 Flash', SCRIPT_BASE / 'baselines_gemini_file_search' / 'Gemini-2.0'),
    ('Gemini 2.5 Flash', SCRIPT_BASE / 'baselines_gemini_file_search' / 'Gemini-2.5'),
    ('LandingAI (ADE)', SCRIPT_BASE / 'baselines_landing_ai_new'),
]

def load_eval_results(trial_dir: Path):
    path = trial_dir / 'evaluation' / 'evaluation_results.json'
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding='utf-8'))

def is_empty(val) -> bool:
    if val is None:
        return True
    s = str(val).strip().lower()
    return s in ('', 'nan', 'not applicable', 'not reported', 'not present', 'na', 'n/a')

# Use PDFs common to all runs
common_pdfs = None
for _, run_path in BASELINE_RUNS:
    if not run_path.exists():
        continue
    pdfs = {d.name for d in run_path.iterdir() if d.is_dir()}
    common_pdfs = pdfs if common_pdfs is None else common_pdfs & pdfs
common_pdfs = sorted(common_pdfs or [])

rows = []
for model_label, run_path in BASELINE_RUNS:
    if not run_path.exists():
        continue
    for pdf in common_pdfs:
        data = load_eval_results(run_path / pdf)
        if data is None:
            continue
        cols = data.get('columns', {})
        empty_gt = 0
        halluc = 0
        for _, col_data in cols.items():
            gt = col_data.get('ground_truth', '')
            pred = col_data.get('predicted', '')
            if is_empty(gt):
                empty_gt += 1
                if not is_empty(pred):
                    halluc += 1
        rate = (halluc / empty_gt) if empty_gt else None
        rows.append({
            'model': model_label,
            'pdf': pdf,
            'hallucination_rate': rate,
        })

df = pd.DataFrame(rows)
pivot = df.pivot(index='pdf', columns='model', values='hallucination_rate')
pivot.loc['AVG'] = pivot.mean(numeric_only=True)
display((pivot * 100).round(1))


model,Gemini 2.0 Flash,Gemini 2.5 Flash,LandingAI (ADE)
pdf,,,
NCT00104715_Gravis_GETUG_EU'15,28.4,95.5,0.0
NCT00268476_Attard_STAMPEDE_Lancet'23,33.3,84.6,11.5
NCT00268476_James_STAMPEDE_IJC'22,23.2,64.3,5.9
NCT00309985_Kriayako_CHAARTED_JCO'18,22.2,58.1,9.4
NCT00309985_Sweeney_CHAARTED_NEJM'15,45.8,62.5,5.7
NCT02799602_Hussain_ARASENS_JCO'23,45.5,77.9,26.3
AVG,33.1,73.8,9.8


In [2]:
import json
from pathlib import Path
import pandas as pd

_candidates = [
    Path.cwd() / 'experiment-scripts',
    (Path.cwd() / '..' / 'experiment-scripts').resolve(),
    Path.cwd(),
]
SCRIPT_BASE = next((p for p in _candidates if p.exists()), None)
if SCRIPT_BASE is None:
    raise FileNotFoundError('Could not find experiment-scripts. Run from repo root or experiment-analysis/.')

BASELINE_RUNS = [
    ('Gemini 2.0 Flash', SCRIPT_BASE / 'baselines_gemini_file_search' / 'Gemini-2.0'),
    ('Gemini 2.5 Flash', SCRIPT_BASE / 'baselines_gemini_file_search' / 'Gemini-2.5'),
    ('LandingAI (ADE)', SCRIPT_BASE / 'baselines_landing_ai_new'),
]

def load_eval_results(trial_dir: Path):
    path = trial_dir / 'evaluation' / 'evaluation_results.json'
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding='utf-8'))

def compute_overall_from_eval(data: dict):
    cols = data.get('columns', {})
    if not cols:
        return None
    correctness = [v.get('correctness', 0) for v in cols.values()]
    completeness = [v.get('completeness', 0) for v in cols.values()]
    overall = [v.get('overall', 0) for v in cols.values()]
    return {
        'avg_correctness': sum(correctness) / len(correctness),
        'avg_completeness': sum(completeness) / len(completeness),
        'avg_overall': sum(overall) / len(overall),
    }

common_pdfs = None
for _, run_path in BASELINE_RUNS:
    if not run_path.exists():
        continue
    pdfs = {d.name for d in run_path.iterdir() if d.is_dir()}
    common_pdfs = pdfs if common_pdfs is None else common_pdfs & pdfs
common_pdfs = sorted(common_pdfs or [])

rows = []
for model_label, run_path in BASELINE_RUNS:
    if not run_path.exists():
        continue
    for pdf in common_pdfs:
        data = load_eval_results(run_path / pdf)
        if data is None:
            continue
        overall = compute_overall_from_eval(data)
        if overall is None:
            continue
        rows.append({
            'model': model_label,
            'pdf': pdf,
            **overall,
        })

df = pd.DataFrame(rows)
pivot = df.pivot(index='pdf', columns='model', values=['avg_correctness','avg_completeness','avg_overall'])
display((pivot * 100).round(1))


avg_correctness                   \
model                                 Gemini 2.0 Flash Gemini 2.5 Flash   
pdf                                                                       
NCT00104715_Gravis_GETUG_EU'15                    66.3             84.4   
NCT00268476_Attard_STAMPEDE_Lancet'23             72.2             72.3   
NCT00268476_James_STAMPEDE_IJC'22                 80.5             89.4   
NCT00309985_Kriayako_CHAARTED_JCO'18              68.2             70.1   
NCT00309985_Sweeney_CHAARTED_NEJM'15              77.5             80.1   
NCT02799602_Hussain_ARASENS_JCO'23                61.7             83.5   

                                                      avg_completeness  \
model                                 LandingAI (ADE) Gemini 2.0 Flash   
pdf                                                                      
NCT00104715_Gravis_GETUG_EU'15                   92.5             65.9   
NCT00268476_Attard_STAMPEDE_Lancet'23            79.3             68.8   
NCT00268476_James_STAMPEDE_IJC'22                87.1             82.0   
NCT00309985_Kriayako_CHAARTED_JCO'18             67.2             65.0   
NCT00309985_Sweeney_CHAARTED_NEJM'15             91.2             77.8   
NCT02799602_Hussain_ARASENS_JCO'23               64.0             61.3   

                                                                        \
model                                 Gemini 2.5 Flash LandingAI (ADE)   
pdf                                                                      
NCT00104715_Gravis_GETUG_EU'15                    82.8            92.1   
NCT00268476_Attard_STAMPEDE_Lancet'23             71.5            77.4   
NCT00268476_James_STAMPEDE_IJC'22                 91.7            86.0   
NCT00309985_Kriayako_CHAARTED_JCO'18              66.8            66.4   
NCT00309985_Sweeney_CHAARTED_NEJM'15              80.1            91.2   
NCT02799602_Hussain_ARASENS_JCO'23                83.5            65.9   

                                           avg_overall                   \
model                                 Gemini 2.0 Flash Gemini 2.5 Flash   
pdf                                                                       
NCT00104715_Gravis_GETUG_EU'15                    66.1             83.6   
NCT00268476_Attard_STAMPEDE_Lancet'23             70.5             71.9   
NCT00268476_James_STAMPEDE_IJC'22                 81.2             90.5   
NCT00309985_Kriayako_CHAARTED_JCO'18              66.6             68.4   
NCT00309985_Sweeney_CHAARTED_NEJM'15              77.6             80.1   
NCT02799602_Hussain_ARASENS_JCO'23                61.5             83.5   

                                                       
model                                 LandingAI (ADE)  
pdf                                                    
NCT00104715_Gravis_GETUG_EU'15                   92.3  
NCT00268476_Attard_STAMPEDE_Lancet'23            78.4  
NCT00268476_James_STAMPEDE_IJC'22                86.6  
NCT00309985_Kriayako_CHAARTED_JCO'18             66.8  
NCT00309985_Sweeney_CHAARTED_NEJM'15             91.2  
NCT02799602_Hussain_ARASENS_JCO'23               65.0

In [3]:
import json
from pathlib import Path
import pandas as pd

_candidates = [
    Path.cwd() / 'experiment-scripts',
    (Path.cwd() / '..' / 'experiment-scripts').resolve(),
    Path.cwd(),
]
SCRIPT_BASE = next((p for p in _candidates if p.exists()), None)
if SCRIPT_BASE is None:
    raise FileNotFoundError('Could not find experiment-scripts. Run from repo root or experiment-analysis/.')

BASELINE_RUNS = [
    ('Gemini 2.0 Flash', SCRIPT_BASE / 'baselines_gemini_file_search' / 'Gemini-2.0'),
    ('Gemini 2.5 Flash', SCRIPT_BASE / 'baselines_gemini_file_search' / 'Gemini-2.5'),
    ('LandingAI (ADE)', SCRIPT_BASE / 'baselines_landing_ai_new'),
]

def load_eval_results(trial_dir: Path):
    path = trial_dir / 'evaluation' / 'evaluation_results.json'
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding='utf-8'))

def numeric_tolerance_score(data: dict):
    cols = data.get('columns', {})
    vals = [v.get('overall', 0) for v in cols.values() if v.get('category') == 'numeric_tolerance']
    if not vals:
        return None
    return sum(vals) / len(vals)

common_pdfs = None
for _, run_path in BASELINE_RUNS:
    if not run_path.exists():
        continue
    pdfs = {d.name for d in run_path.iterdir() if d.is_dir()}
    common_pdfs = pdfs if common_pdfs is None else common_pdfs & pdfs
common_pdfs = sorted(common_pdfs or [])

rows = []
for model_label, run_path in BASELINE_RUNS:
    if not run_path.exists():
        continue
    for pdf in common_pdfs:
        data = load_eval_results(run_path / pdf)
        if data is None:
            continue
        score = numeric_tolerance_score(data)
        rows.append({
            'model': model_label,
            'pdf': pdf,
            'numeric_tolerance_avg_overall': score,
        })

df = pd.DataFrame(rows)
pivot = df.pivot(index='pdf', columns='model', values='numeric_tolerance_avg_overall')
pivot.loc['AVG'] = pivot.mean(numeric_only=True)
display((pivot * 100).round(1))


model,Gemini 2.0 Flash,Gemini 2.5 Flash,LandingAI (ADE)
pdf,,,
NCT00104715_Gravis_GETUG_EU'15,64.2,83.3,96.3
NCT00268476_Attard_STAMPEDE_Lancet'23,68.2,71.4,82.2
NCT00268476_James_STAMPEDE_IJC'22,84.3,91.5,90.6
NCT00309985_Kriayako_CHAARTED_JCO'18,64.5,64.5,68.1
NCT00309985_Sweeney_CHAARTED_NEJM'15,79.5,83.6,95.2
NCT02799602_Hussain_ARASENS_JCO'23,63.1,82.2,68.4
AVG,70.6,79.4,83.5
